# DDR Chart Generator — Training Notebook

**Before running: Runtime → Change runtime type → T4 GPU**

### Assumed setup
Your Google Drive already has StepMania packs uploaded at:
```
MyDrive/ddc_project/data/raw/
  Mister Green Bean's Anime.../
    Song Name/
      Song.sm
      Song.mp3
  Some Other Pack/
    ...
```
The folder names don't matter — the code searches recursively for any folder
that contains both a .sm file and an audio file (.mp3 / .ogg / .wav).

### Run order (every session)
Cells 1-3 must be re-run every session (Colab VMs are wiped).
Cells 4+ only need to run once — results are saved to Drive.

In [ ]:
# ── 1. Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Create output folders if they don't exist yet
!mkdir -p /content/drive/MyDrive/ddc_project/data/cache
!mkdir -p /content/drive/MyDrive/ddc_project/checkpoints
!mkdir -p /content/drive/MyDrive/ddc_project/logs
print('Drive mounted.')

In [ ]:
# ── 2. Clone your project repo ──────────────────────────────────────────
# Replace with your actual GitHub repo URL
%cd /content
!git clone https://github.com/YOUR_USERNAME/ddc-chart-gen.git
%cd /content/ddc-chart-gen

# Symlink Drive data into the repo so code can find it at 'data/'
!ln -sf /content/drive/MyDrive/ddc_project/data data
print('Repo cloned and data symlinked.')

In [ ]:
# ── 3. Install dependencies ─────────────────────────────────────────────
!pip install librosa soundfile torchaudio --quiet
print('Done.')

In [ ]:
# ── 4. Verify data ───────────────────────────────────────────────────────
# Confirms the code can see your uploaded packs and parse them correctly.
import sys
sys.path.insert(0, '/content/ddc-chart-gen')

from pathlib import Path
from utils.data_utils import build_sample, parse_sm_file

DATA_ROOT = '/content/drive/MyDrive/ddc_project/data/raw'

all_sm    = list(Path(DATA_ROOT).rglob('*.sm'))
all_audio = [f for f in Path(DATA_ROOT).rglob('*')
             if f.suffix.lower() in {'.ogg', '.mp3', '.wav'}]
print(f'Found {len(all_sm)} .sm files and {len(all_audio)} audio files')

# Count how many songs have both .sm AND audio (usable for training)
usable = []
for sm in all_sm:
    audio = [f for f in sm.parent.iterdir()
             if f.suffix.lower() in {'.ogg', '.mp3', '.wav'}]
    if audio:
        usable.append((sm, audio[0]))
print(f'Usable song-audio pairs: {len(usable)}')

# Parse one song as a sanity check
sm_path, audio_path = usable[0]
sm_data = parse_sm_file(str(sm_path))
print(f'\nSample song: {sm_data["title"]}')
print(f'Charts: {[(c["difficulty"], c["meter"]) for c in sm_data["charts"]]}')

sample = build_sample(str(audio_path), str(sm_path))
if sample:
    print(f'X shape: {sample["X"].shape}  (timesteps, context_window, mel_bins)')
    print(f'y shape: {sample["y"].shape}  (timesteps, 4 arrows)')
    print(f'Step density: {(sample["y"].sum(-1) > 0).mean():.1%} of timesteps have a step')
else:
    print('build_sample returned None — check the .sm file has a dance-single chart')

In [ ]:
# ── 5. Train ─────────────────────────────────────────────────────────────
# Set epochs_per_stage=5 for a quick smoke test first.
# Set epochs_per_stage=20 for a real training run.
#
# Curriculum stages 0->4 (beginner -> challenge).
# Each stage trains for epochs_per_stage epochs then moves to the next difficulty.
# Checkpoints and cache are saved to Drive so you can resume if Colab disconnects.

DATA_ROOT = '/content/drive/MyDrive/ddc_project/data/raw'
CACHE_DIR = '/content/drive/MyDrive/ddc_project/data/cache'
CKPT_DIR  = '/content/drive/MyDrive/ddc_project/checkpoints'

%cd /content/ddc-chart-gen
!python train.py \
    --data_root  "{DATA_ROOT}" \
    --cache_dir  "{CACHE_DIR}" \
    --checkpoint_dir "{CKPT_DIR}" \
    --epochs_per_stage 20 \
    --batch_size 32 \
    --d_model 256 \
    --n_layers 4 \
    --curriculum_start 0

In [ ]:
# ── 6. Plot loss curves ──────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

with open('logs/train_history.json') as f:
    history = json.load(f)

train_h = history['train']
val_h   = history['val']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
stage_colors = {0:'#3498db', 1:'#2ecc71', 2:'#f39c12', 3:'#e74c3c', 4:'#9b59b6'}
stage_names  = {0:'Beginner', 1:'Easy', 2:'Medium', 3:'Hard', 4:'Challenge'}

for metric, ax, title in [
    ('loss',       axes[0], 'Total Loss'),
    ('step_loss',  axes[1], 'Step Placement Loss'),
    ('f1',         axes[2], 'Step F1 Score'),
]:
    for stage in range(5):
        t_vals = [x[metric] for x in train_h if x['stage'] == stage]
        v_vals = [x[metric] for x in val_h   if x['stage'] == stage]
        if not t_vals:
            continue
        color = stage_colors[stage]
        ax.plot(t_vals, color=color, alpha=0.9, label=f'{stage_names[stage]} train')
        ax.plot(v_vals, color=color, alpha=0.5, linestyle='--', label=f'{stage_names[stage]} val')
    ax.set_title(title)
    ax.set_xlabel('Epoch within stage')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle('Training curves by curriculum stage', y=1.02)
plt.tight_layout()
plt.savefig('logs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
!cp logs/training_curves.png /content/drive/MyDrive/ddc_project/logs/
print('Saved to Drive.')

In [ ]:
# ── 7. Generate a chart ──────────────────────────────────────────────────
# Uses the first song from your dataset by default.
# To use your own song instead, uncomment Option B.

from pathlib import Path

# Option A: pick a song already in your dataset
DATA_ROOT = '/content/drive/MyDrive/ddc_project/data/raw'
all_sm = list(Path(DATA_ROOT).rglob('*.sm'))
for sm in all_sm:
    audio = [f for f in sm.parent.iterdir()
             if f.suffix.lower() in {'.ogg', '.mp3', '.wav'}]
    if audio:
        test_audio = str(audio[0])
        print(f'Using: {sm.parent.name} — {audio[0].name}')
        break

# Option B: upload your own song
# from google.colab import files as colab_files
# uploaded = colab_files.upload()
# test_audio = list(uploaded.keys())[0]

CKPT_DIR = '/content/drive/MyDrive/ddc_project/checkpoints'
!python generate.py \
    --audio     "{test_audio}" \
    --checkpoint "{CKPT_DIR}/best_model.pt" \
    --difficulty 2 \
    --threshold  0.5 \
    --output     output_chart

!cp -r output_chart /content/drive/MyDrive/ddc_project/
print('Chart saved to Drive.')

In [ ]:
# ── 8. Threshold sweep ───────────────────────────────────────────────────
# Shows how the step threshold knob controls chart density.
# Lower threshold = more arrows = harder feel.
# Higher threshold = fewer arrows = easier feel.

import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/content/ddc-chart-gen')

from models.model import DDRTransformer, generate_chart
from generate import audio_to_model_input

CKPT_DIR = '/content/drive/MyDrive/ddc_project/checkpoints'
ckpt = torch.load(f'{CKPT_DIR}/best_model.pt', map_location='cpu')
model_args = ckpt.get('args', {})
model = DDRTransformer(
    d_model=model_args.get('d_model', 256),
    nhead=model_args.get('nhead', 8),
    num_encoder_layers=model_args.get('n_layers', 4),
    dim_feedforward=model_args.get('d_ff', 1024),
    dropout=0.0,
)
model.load_state_dict(ckpt['model_state'])
X = audio_to_model_input(test_audio)

thresholds = np.linspace(0.1, 0.9, 17)
densities  = []
for th in thresholds:
    mask, _ = generate_chart(model, X, difficulty=2, step_threshold=float(th))
    densities.append(mask.mean())

plt.figure(figsize=(7, 4))
plt.plot(thresholds, densities, 'o-', color='#e74c3c')
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='default threshold')
plt.xlabel('Step threshold')
plt.ylabel('Fraction of timesteps with a step')
plt.title('Difficulty knob: threshold vs. chart density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('logs/threshold_sweep.png', dpi=150)
plt.show()
!cp logs/threshold_sweep.png /content/drive/MyDrive/ddc_project/logs/

In [ ]:
# ── 9. Download results to your local machine ────────────────────────────
# Results are already on Drive — this just downloads them directly to your computer.
from google.colab import files

files.download('output_chart/visualizer.html')       # open in browser to view chart
files.download('output_chart/chart.sm')              # load in StepMania
files.download(f'{CKPT_DIR}/best_model.pt')          # trained model weights
files.download('logs/training_curves.png')           # loss curves
files.download('logs/threshold_sweep.png')           # difficulty knob plot